# Lab 1 — MCP Fundamentals: Helios's Diligence Glossary Server

**Scenario.** Solstice Capital Partners is a mid-market infrastructure investor that acquires
operating and late-stage-development solar, wind, and storage projects across the US. Its
investment analysts spend a lot of time relearning the same jargon — capacity factor, curtailment,
merchant tail risk — every time they open a new deal. The platform engineering team is building
**Helios**, an internal copilot that gives analysts instant, sourced answers during diligence.

This notebook builds the *first* piece of Helios: a diligence glossary exposed as a real **MCP
server**, called by a **LangGraph** agent. Everything here is a genuine client/server round trip
over the **stdio transport** — no shortcuts that just call a Python function directly — so you can
see exactly what MCP adds on top of "import a function and call it."

## What you'll build
1. A small glossary knowledge base (the *data* Helios searches).
2. `glossary_server.py` — a `FastMCP` server exposing a **tool**, a **resource**, and a **prompt**.
3. A raw MCP client session (`ClientSession` over `stdio_client`) that talks to that server directly.
4. A LangChain-compatible tool wrapper, so the same server can be handed to a LangGraph agent.
5. A ReAct agent (Groq if you have a key, a clearly-labeled offline fallback if you don't).

## Concept map
- **What MCP solves** — a standard interface so any host (IDE, notebook, agent) can talk to any
  server (database, filesystem, SaaS API) without a bespoke integration for every pair.
- **Host / client / server** — *this notebook* is the host. It creates an MCP *client*, which
  launches and talks to the glossary *server* as a separate OS process.
- **Transport** — the current MCP Python SDK's primary transports are **stdio** (one local
  subprocess per client, used in this lab) and **Streamable HTTP** (a remote server many clients
  can share, used in Lab 3's production notes). Older ecosystem examples reference SSE, which is
  legacy.
- **Primitives** — `@mcp.tool()` (callable, side-effecting or not), `@mcp.resource()` (readable,
  addressed by a URI), `@mcp.prompt()` (a reusable prompt template the *client* can request).
- **MCP Inspector** — the standard way to poke at a server by hand before wiring it into an agent.

In [1]:
# --- Single setup cell: dependencies, imports, and paths -------------------
%pip install --quiet --break-system-packages "mcp>=1.8.0" "langgraph>=0.2.0" "langchain>=0.3.0" \
    "langchain-core" "langchain-groq>=0.2.0" "python-dotenv>=1.0.1" "pydantic>=2.7.0"

import os
from pathlib import Path

from dotenv import load_dotenv

# Picks up a .env file if you create one next to this notebook. Optional --
# every cell below works with or without it.
load_dotenv()

# Everything this notebook creates lives next to the notebook itself, so it
# works regardless of where the project is checked out.
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SERVERS_DIR = ROOT / "mcp_servers"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SERVERS_DIR.mkdir(parents=True, exist_ok=True)

GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

print(f"ROOT         = {ROOT}")
print(f"DATA_DIR     = {DATA_DIR}")
print(f"SERVERS_DIR  = {SERVERS_DIR}")
print(f"GROQ_API_KEY set: {bool(os.getenv('GROQ_API_KEY'))}  (model if set: {GROQ_MODEL})")

Note: you may need to restart the kernel to use updated packages.


ROOT         = /home/claude/build/run/deliver/lab1
DATA_DIR     = /home/claude/build/run/deliver/lab1/data
SERVERS_DIR  = /home/claude/build/run/deliver/lab1/mcp_servers
GROQ_API_KEY set: False  (model if set: openai/gpt-oss-20b)


## Step 1 — The knowledge base Helios will search

Before building any MCP plumbing, define the data. This is Solstice's internal diligence
glossary: the terms analysts look up constantly while reviewing a new acquisition target. In
production this would live in Confluence or a vector database; here it's a plain Python list so
the lab stays dependency-light. The *scoring* function below is a simple token-overlap ranker —
good enough for a glossary lookup tool, and deliberately simple so the MCP wiring stays the star
of this lab. (Lab 3 upgrades to real embedding-based retrieval for unstructured PDF search.)

In [2]:
# --- Helios diligence glossary -------------------------------------------
GLOSSARY: list[dict[str, str]] = [
    {
        "term": "Power Purchase Agreement (PPA)",
        "summary": "A long-term contract where a buyer (utility, corporate offtaker, or trading "
        "desk) agrees to purchase a project's electricity at a set price or formula for a fixed "
        "term, usually 10-20 years. The PPA is the primary revenue certainty lenders and "
        "investors underwrite against.",
    },
    {
        "term": "Capacity Factor",
        "summary": "Actual energy produced over a period divided by the theoretical maximum if "
        "the plant ran at full nameplate capacity the whole time. Solar in the US Southwest "
        "often runs 25-32%; wind ranges roughly 30-45% depending on site quality.",
    },
    {
        "term": "Curtailment",
        "summary": "When a grid operator or the project itself reduces output below available "
        "capacity, usually because of transmission congestion, oversupply, or a plant-level "
        "constraint. Curtailment directly reduces revenue and is a key diligence risk item in "
        "congested interconnection zones.",
    },
    {
        "term": "Interconnection Queue",
        "summary": "The ordered backlog of generation projects waiting for a transmission "
        "operator to study and approve their grid connection. Queue position and the resulting "
        "network upgrade cost estimate are major schedule and cost risks for development-stage "
        "projects.",
    },
    {
        "term": "Investment Tax Credit (ITC)",
        "summary": "A US federal tax credit equal to a percentage of a project's eligible "
        "capital cost, claimed once the project reaches commercial operation. Bonus adders "
        "exist for domestic content, energy communities, and low-income siting.",
    },
    {
        "term": "Production Tax Credit (PTC)",
        "summary": "A US federal tax credit paid per MWh generated during a project's first 10 "
        "years of operation, historically the default structure for wind. Developers choose "
        "ITC or PTC per project based on capital intensity and expected output.",
    },
    {
        "term": "Debt Service Coverage Ratio (DSCR)",
        "summary": "Cash available for debt service divided by the debt payment due in a period. "
        "Project-finance lenders set a minimum DSCR covenant (often 1.20-1.35x); breaching it can "
        "trigger a cash sweep or default.",
    },
    {
        "term": "Merchant Tail Risk",
        "summary": "The revenue uncertainty in the years after a project's PPA expires, when it "
        "sells into the wholesale market at whatever price prevails. Longer merchant tails "
        "generally lower an asset's valuation because future cash flows are less certain.",
    },
    {
        "term": "Commercial Operation Date (COD)",
        "summary": "The date a project is certified able to reliably deliver power, triggering "
        "PPA revenue, tax credit eligibility, and (for a construction-stage acquisition) the "
        "final diligence milestone before funds are released.",
    },
    {
        "term": "Degradation Rate",
        "summary": "The annual decline in a solar module's power output over its lifetime, "
        "typically 0.4-0.7% per year. Independent Engineers apply a degradation curve when "
        "forecasting 20-30 year energy yield for financial models.",
    },
    {
        "term": "Tax Equity Partnership",
        "summary": "A financing structure where an investor with US tax liability contributes "
        "capital in exchange for the project's tax credits and accelerated depreciation, while "
        "the developer retains most of the cash and long-term ownership. Common for both ITC and "
        "PTC projects.",
    },
    {
        "term": "Basis Risk",
        "summary": "The mismatch between the price at a project's grid node and the hub or index "
        "price referenced in its PPA or hedge. Wide or volatile basis erodes the value of an "
        "otherwise well-priced offtake contract.",
    },
    {
        "term": "Independent Engineer (IE) Report",
        "summary": "A third-party technical due-diligence report covering design, construction "
        "quality, resource assessment, and projected performance. Lenders and investors typically "
        "require a fresh or updated IE report before closing.",
    },
    {
        "term": "Notice to Proceed (NTP)",
        "summary": "The formal milestone authorizing construction to begin, usually gated on "
        "financing close, permits in hand, and a signed interconnection agreement. Projects are "
        "often bucketed by stage as pre-NTP, under construction, or operating.",
    },
    {
        "term": "Locational Marginal Price (LMP) / Congestion",
        "summary": "The real-time or day-ahead price of energy at a specific grid location. LMP "
        "can diverge sharply from the broader hub price when transmission congestion limits how "
        "much power can flow out of a region -- a key driver of both curtailment and basis risk.",
    },
]


def _tokenize(text: str) -> set[str]:
    return {t.lower().strip(".,:;()[]{}") for t in text.replace("_", " ").replace("-", " ").split() if len(t) > 2}


def _score(query: str, text: str) -> int:
    return len(_tokenize(query) & _tokenize(text))


def search_glossary_corpus(query: str, top_k: int = 3) -> list[dict]:
    """Rank glossary entries against a query by token overlap. Same shape whether called
    directly (this cell) or through the MCP tool defined a few cells down."""
    ranked = sorted(
        (
            {"term": e["term"], "summary": e["summary"], "score": _score(query, e["term"] + " " + e["summary"])}
            for e in GLOSSARY
        ),
        key=lambda item: item["score"],
        reverse=True,
    )
    return ranked[: max(1, top_k)]


def format_matches(matches: list[dict]) -> str:
    return "\n".join(f"{i}. {m['term']} (score={m['score']}): {m['summary']}" for i, m in enumerate(matches, 1))


def glossary_markdown() -> str:
    lines = ["# Solstice diligence glossary"]
    lines += [f"- **{e['term']}** -- {e['summary']}" for e in GLOSSARY]
    return "\n".join(lines)


print(f"Loaded {len(GLOSSARY)} glossary terms.")
print(format_matches(search_glossary_corpus("What happens when a grid operator curtails a solar project?", top_k=2)))

Loaded 15 glossary terms.
1. Curtailment (score=3): When a grid operator or the project itself reduces output below available capacity, usually because of transmission congestion, oversupply, or a plant-level constraint. Curtailment directly reduces revenue and is a key diligence risk item in congested interconnection zones.
2. Interconnection Queue (score=2): The ordered backlog of generation projects waiting for a transmission operator to study and approve their grid connection. Queue position and the resulting network upgrade cost estimate are major schedule and cost risks for development-stage projects.


In [3]:
# --- Give the (separate-process) server its own data source ---------------
# A real MCP server is a standalone process -- it can't reach into this
# notebook's Python memory. We persist the glossary to disk once, and the
# server script we write in the next cell reads it from there, the same way
# it might read from a config file or a database table in production.
import json

glossary_path = DATA_DIR / "glossary.json"
glossary_path.write_text(json.dumps(GLOSSARY, indent=2), encoding="utf-8")
print(f"Wrote {len(GLOSSARY)} terms to {glossary_path}")

Wrote 15 terms to /home/claude/build/run/deliver/lab1/data/glossary.json


## Step 2 — Write the MCP server

`%%writefile` below saves a standalone Python script to `mcp_servers/glossary_server.py`. This is
a real, runnable MCP server -- you could execute it directly with `python glossary_server.py`, or
add it to any MCP-compatible host's config. It exposes three kinds of MCP primitive:

- `search_glossary(query, top_k)` -- a **tool**: the model decides when to call it and with what
  arguments.
- `glossary://map` -- a **resource**: static-ish content addressed by URI, meant to be attached to
  context rather than "called."
- `search_prompt(topic, audience)` -- a **prompt**: a parameterized template the *client* can pull
  and fill in, so prompt engineering can live in the server instead of being copy-pasted across
  every client that uses it.

In [4]:
%%writefile mcp_servers/glossary_server.py
"""Helios glossary MCP server.

Standalone process: run directly with `python glossary_server.py`, inspect it
with the MCP Inspector, or launch it from a client over stdio. It reads its
data from data/glossary.json rather than sharing memory with any notebook.
"""
from __future__ import annotations

import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

GLOSSARY_PATH = Path("data/glossary.json")
GLOSSARY: list[dict] = json.loads(GLOSSARY_PATH.read_text(encoding="utf-8"))

mcp = FastMCP("helios-glossary")


def _tokenize(text: str) -> set[str]:
    return {t.lower().strip(".,:;()[]{}") for t in text.replace("_", " ").replace("-", " ").split() if len(t) > 2}


def _score(query: str, text: str) -> int:
    return len(_tokenize(query) & _tokenize(text))


@mcp.tool()
def search_glossary(query: str, top_k: int = 3) -> dict:
    """Search Solstice's renewable-energy diligence glossary and return the top matching terms."""
    ranked = sorted(
        (
            {"term": e["term"], "summary": e["summary"], "score": _score(query, e["term"] + " " + e["summary"])}
            for e in GLOSSARY
        ),
        key=lambda item: item["score"],
        reverse=True,
    )
    matches = ranked[: max(1, top_k)]
    rendered = "\n".join(f"{i}. {m['term']} (score={m['score']}): {m['summary']}" for i, m in enumerate(matches, 1))
    return {"query": query, "top_k": top_k, "matches": matches, "rendered": rendered}


@mcp.resource("glossary://map")
def glossary_map() -> str:
    """Return the full glossary as markdown -- meant to be attached as context, not called like a tool."""
    lines = ["# Solstice diligence glossary"]
    lines += [f"- **{e['term']}** -- {e['summary']}" for e in GLOSSARY]
    return "\n".join(lines)


@mcp.prompt()
def search_prompt(topic: str, audience: str = "analyst") -> str:
    """Build a retrieval-and-explain prompt template for a given topic and audience."""
    return (
        f"You are briefing a {audience} on {topic} during live diligence. "
        "Look it up in the glossary, then explain in three short parts: what it means, "
        "why it matters for underwriting this deal, and one concrete example."
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing mcp_servers/glossary_server.py


## Optional — poke at it with MCP Inspector

The Inspector is the fastest way to explore a stdio server by hand, before writing any client
code. It needs Node.js. From a terminal in this notebook's directory:

```bash
npx @modelcontextprotocol/inspector python mcp_servers/glossary_server.py
```

This opens a local web UI where you can call `search_glossary`, read the `glossary://map`
resource, and fetch the `search_prompt` template -- useful for sanity-checking a server before any
agent ever touches it. The next cell does the same three things programmatically.

## Step 3 — Talk to it as a raw MCP client

This is the part the original "just call the Python function" shortcut skips. `stdio_client`
launches `glossary_server.py` as a **child process** and opens a JSON-RPC channel over its
stdin/stdout; `ClientSession` speaks the MCP handshake on top of that. Every call below crosses a
real process boundary -- kill this notebook's kernel and the server process goes with it, exactly
like a production host/server pair.

In [5]:
# --- Raw MCP client session (host -> client -> server, for real) ----------
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

GLOSSARY_SERVER = StdioServerParameters(
    command="python3",
    args=[str(SERVERS_DIR / "glossary_server.py")],
    cwd=str(ROOT),  # so the server's relative "data/glossary.json" resolves correctly
)


async def demo_raw_session() -> None:
    async with stdio_client(GLOSSARY_SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()  # the MCP handshake

            tools = await session.list_tools()
            print("tools:", [t.name for t in tools.tools])

            resources = await session.list_resources()
            print("resources:", [str(r.uri) for r in resources.resources])

            prompts = await session.list_prompts()
            print("prompts:", [p.name for p in prompts.prompts])

            print("\n--- call_tool('search_glossary') ---")
            result = await session.call_tool("search_glossary", {"query": "curtailment risk", "top_k": 2})
            print(result.content[0].text)

            print("\n--- read_resource('glossary://map') (first 200 chars) ---")
            res = await session.read_resource("glossary://map")
            print(res.contents[0].text[:200], "...")

            print("\n--- get_prompt('search_prompt') ---")
            prompt = await session.get_prompt("search_prompt", {"topic": "merchant tail risk"})
            print(prompt.messages[0].content.text)


await demo_raw_session()

tools: ['search_glossary']
resources: ['glossary://map']
prompts: ['search_prompt']

--- call_tool('search_glossary') ---
{
  "query": "curtailment risk",
  "top_k": 2,
  "matches": [
    {
      "term": "Curtailment",
      "summary": "When a grid operator or the project itself reduces output below available capacity, usually because of transmission congestion, oversupply, or a plant-level constraint. Curtailment directly reduces revenue and is a key diligence risk item in congested interconnection zones.",
      "score": 2
    },
    {
      "term": "Locational Marginal Price (LMP) / Congestion",
      "summary": "The real-time or day-ahead price of energy at a specific grid location. LMP can diverge sharply from the broader hub price when transmission congestion limits how much power can flow out of a region -- a key driver of both curtailment and basis risk.",
      "score": 2
    }
  ],
  "rendered": "1. Curtailment (score=2): When a grid operator or the project itself reduces out

## Step 4 — Wrap the MCP tool so LangGraph can call it

A LangGraph agent expects LangChain `Tool` objects, not raw MCP sessions. The wrapper below opens
a fresh client session per call -- simple and correct for a low-traffic lab, though a production
host would keep a long-lived session (or a pool) instead of paying the subprocess-startup cost on
every single tool call. Lab 3's `MultiServerMCPClient` is the library-level answer to that; here
you're seeing what it does under the hood.

In [6]:
# --- Adapt the MCP tool into a LangChain-compatible tool -------------------
from langchain_core.tools import tool


@tool
async def search_glossary_tool(query: str, top_k: int = 3) -> str:
    """Search Solstice's renewable-energy diligence glossary for a query and return the top matches."""
    async with stdio_client(GLOSSARY_SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("search_glossary", {"query": query, "top_k": top_k})
            return result.content[0].text


# Quick direct test -- no LLM involved yet, just confirming the wrapper works.
print(await search_glossary_tool.ainvoke({"query": "difference between ITC and PTC", "top_k": 2}))

{
  "query": "difference between ITC and PTC",
  "top_k": 2,
  "matches": [
    {
      "term": "Production Tax Credit (PTC)",
      "summary": "A US federal tax credit paid per MWh generated during a project's first 10 years of operation, historically the default structure for wind. Developers choose ITC or PTC per project based on capital intensity and expected output.",
      "score": 3
    },
    {
      "term": "Tax Equity Partnership",
      "summary": "A financing structure where an investor with US tax liability contributes capital in exchange for the project's tax credits and accelerated depreciation, while the developer retains most of the cash and long-term ownership. Common for both ITC and PTC projects.",
      "score": 3
    }
  ],
  "rendered": "1. Production Tax Credit (PTC) (score=3): A US federal tax credit paid per MWh generated during a project's first 10 years of operation, historically the default structure for wind. Developers choose ITC or PTC per project based 

## Step 5 — Put an agent in front of the tool

The MCP round trip above works with or without an LLM -- it's just a function call over a
process boundary. What an LLM adds is *deciding* whether and how to call it, and turning the raw
match back into a natural-language answer. That decision layer is where this notebook needs
Groq, so it's the one part with a genuine online/offline split:

- **`GROQ_API_KEY` set** -> a real LangGraph `create_react_agent` reasons over the tool.
- **not set** -> a small `[offline-fallback]` routine calls the same MCP tool directly and
  templates an answer from the top match, so the notebook still runs end to end with no network
  access at all. Both paths are exercised through the same `ask_helios(question)` function below.

In [7]:
# --- Groq if available, otherwise a clearly-labeled offline fallback ------
def get_groq_llm(temperature: float = 0.0):
    """Return a ChatGroq model if GROQ_API_KEY is set and reachable, else None.
    Callers branch on None to run their [offline-fallback] path -- same pattern used
    throughout this curriculum so every notebook runs top-to-bottom with no network access."""
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        return None
    try:
        from langchain_groq import ChatGroq

        return ChatGroq(model=GROQ_MODEL, temperature=temperature, api_key=api_key)
    except Exception as e:  # network blocked, bad key, package mismatch, etc.
        print(f"[offline-fallback] Could not initialize Groq ({e}); falling back.")
        return None


LLM = get_groq_llm()
print("Using live Groq model" if LLM is not None else "[offline-fallback] No GROQ_API_KEY -- using offline responder")

[offline-fallback] No GROQ_API_KEY -- using offline responder


In [8]:
# --- Build the agent (or its offline stand-in) -----------------------------
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = (
    "You are Helios, Solstice Capital Partners' diligence copilot. Use the glossary search tool "
    "whenever a question touches renewable-energy or project-finance jargon. Keep answers to two "
    "or three sentences and cite the term you looked up."
)

if LLM is not None:
    agent = create_react_agent(LLM, tools=[search_glossary_tool], prompt=SYSTEM_PROMPT)

    async def ask_helios(question: str) -> str:
        result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
        return result["messages"][-1].content

else:
    import json

    async def ask_helios(question: str) -> str:
        """[offline-fallback] Skips the LLM reasoning step: calls the MCP tool directly with the
        raw question, then templates an answer from the top glossary match. Lower quality than a
        real agent (no multi-hop reasoning, no query rewriting) but demonstrates the same
        question-in / answer-out contract with zero network access.

        Note: the MCP tool returns its payload as a JSON *string* (FastMCP serializes the dict
        return value before it crosses the process boundary) -- an earlier draft of this cell
        sliced that raw string directly and printed a stray "{" instead of the match. Parsing it
        back into a dict first is the fix."""
        raw = await search_glossary_tool.ainvoke({"query": question, "top_k": 1})
        try:
            payload = json.loads(raw)
            top = payload["matches"][0]
            answer = f"{top['term']} -- {top['summary']}"
        except (json.JSONDecodeError, KeyError, IndexError):
            answer = raw
        return f"[offline-fallback] {answer}"


print("Agent ready:", "live Groq ReAct agent" if LLM is not None else "[offline-fallback] direct tool responder")

Agent ready: [offline-fallback] direct tool responder


In [9]:
# --- Run a few real analyst questions through Helios -----------------------
DEMO_QUESTIONS = [
    "What's the practical difference between the ITC and the PTC?",
    "Why would curtailment matter more in a congested interconnection zone?",
    "What does a lender's DSCR covenant actually protect them against?",
    "Explain merchant tail risk to a first-year analyst.",
]

for q in DEMO_QUESTIONS:
    print(f"\nQ: {q}")
    print(await ask_helios(q))


Q: What's the practical difference between the ITC and the PTC?


[offline-fallback] Investment Tax Credit (ITC) -- A US federal tax credit equal to a percentage of a project's eligible capital cost, claimed once the project reaches commercial operation. Bonus adders exist for domestic content, energy communities, and low-income siting.

Q: Why would curtailment matter more in a congested interconnection zone?


[offline-fallback] Curtailment -- When a grid operator or the project itself reduces output below available capacity, usually because of transmission congestion, oversupply, or a plant-level constraint. Curtailment directly reduces revenue and is a key diligence risk item in congested interconnection zones.

Q: What does a lender's DSCR covenant actually protect them against?


[offline-fallback] Debt Service Coverage Ratio (DSCR) -- Cash available for debt service divided by the debt payment due in a period. Project-finance lenders set a minimum DSCR covenant (often 1.20-1.35x); breaching it can trigger a cash sweep or default.

Q: Explain merchant tail risk to a first-year analyst.


[offline-fallback] Merchant Tail Risk -- The revenue uncertainty in the years after a project's PPA expires, when it sells into the wholesale market at whatever price prevails. Longer merchant tails generally lower an asset's valuation because future cash flows are less certain.


## Verification

| Item | Status |
|---|---|
| Glossary data + token-overlap search | Sandbox-verified (pure Python, runs every time) |
| `glossary_server.py` written and spawned as a subprocess | Sandbox-verified |
| Raw `ClientSession` — `list_tools` / `call_tool` / `read_resource` / `get_prompt` | Sandbox-verified, real stdio round trip |
| `search_glossary_tool` LangChain wrapper | Sandbox-verified |
| ReAct agent via `ChatGroq` | Written against the current `langgraph.prebuilt` API; needs a real `GROQ_API_KEY` to execute live — this notebook ran the `[offline-fallback]` path |
| MCP Inspector walkthrough | Manual step outside the notebook — needs Node.js, not run automatically |

## Edit points

- Add or rewrite entries in the `GLOSSARY` list defined earlier in this notebook to cover the
  terms your own deals care about, then re-run the export cell so `glossary_server.py` picks up
  the change.
- Swap the token-overlap `_score` function inside `mcp_servers/glossary_server.py` for a real
  embedding similarity search once your glossary grows past a hundred terms or so.
- Add a second `@mcp.tool()` to `glossary_server.py` -- e.g. `define_acronym(acronym)` -- and
  re-run the raw client cell to see it show up in `list_tools()` with no other changes needed.
- Point `GROQ_MODEL` at `openai/gpt-oss-120b` (set it before the setup cell runs) if you want
  higher-quality reasoning at the cost of latency.

**Next:** `lab2_sqlite_mcp_solstice.ipynb` puts Helios's structured project tracker behind MCP.